# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

print("Token loaded and login successful!")

Token loaded and login successful!


In [ ]:
from huggingface_hub import HfFileSystem

fs = HfFileSystem()
files = fs.ls("datasets/FlyRank/internship-warehouse", detail=False)
for f in files:
    print(f)



files = fs.ls("datasets/FlyRank/internship-warehouse/fact_content_daily_performance", detail=False)
for f in files:
    print(f)



datasets/FlyRank/internship-warehouse/fact_content_daily_performance
datasets/FlyRank/internship-warehouse/.gitattributes
datasets/FlyRank/internship-warehouse/README.md
datasets/FlyRank/internship-warehouse/dim_clients.parquet
datasets/FlyRank/internship-warehouse/dim_content.parquet
datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet
datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06
datasets/FlyRank/internship-warehouse/fact_content_daily_perfor

In [ ]:
files = fs.ls("datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03", detail=False)
for f in files:
    print(f)

datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet


In [2]:
import pandas as pd

url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
df_march = pd.read_parquet(url)

print("Shape:", df_march.shape)
print("Date range:", df_march["report_date"].min(), "to", df_march["report_date"].max())
df_march.head()

print(df_march.columns.tolist())

Shape: (9841378, 31)
Date range: 2026-03-01 to 2026-03-31
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In fact_content_daily_performance, one row represents one page, for one client, on one specific day  not a single overall snapshot like in the starter CSV. I proved this by checking one page (content_id: content_b7e512995f79d5a6) for the client client_73cda7b4e4f265ea across March 2026: it had exactly 31 rows, one for each day of the month. This confirms the table works like a daily diary every page gets a fresh entry each day it's tracked, rather than one lifetime total. This matters for my lane because any feature I build (like average CTR or engagement) needs to be calculated across these daily rows for a page, not read off a single row.

I chose March 2026 as my working month specifically because the assignment warns that the _sample table is not a random sample  it is exactly June 2026, the sealed final month reserved for later testing. Building features from June now would risk leaking information from the outcome period into my analysis. March 2026 is a safe, ordinary mid-panel month instead, with no special role in the data.

In [ ]:
# Pick ONE page and see how many rows it has in March
sample_content = df_march["content_hash_id"].iloc[0]
sample_client = df_march["client_hash_id"].iloc[0]

rows_for_this_page = df_march[
    (df_march["content_hash_id"] == sample_content) &
    (df_march["client_hash_id"] == sample_client)
]

print("Page ID:", sample_content)
print("Number of rows for this one page in March:", rows_for_this_page.shape[0])
rows_for_this_page[["report_date", "client_hash_id", "content_hash_id"]]

Page ID: content_b7e512995f79d5a6
Number of rows for this one page in March: 31


,report_date,client_hash_id,content_hash_id
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
555866,2026-03-03,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1334822,2026-03-04,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1499377,2026-03-05,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1573240,2026-03-07,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1594884,2026-03-02,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
2009723,2026-03-08,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
2281373,2026-03-06,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
2439321,2026-03-09,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
2553195,2026-03-10,client_73cda7b4e4f265ea,content_b7e512995f79d5a6


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

I sorted every column in fact_content_daily_performance into 4 buckets. Context columns (report_date, client_hash_id, content_hash_id, month, and the four availability flags) are used only for grouping, joining, and filtering — never as model inputs, since IDs are pseudonyms per the data dictionary.

Feature columns are the remaining raw, independent signals (impressions, position, sessions, channel breakdowns, scroll events).

My label doesn't exist as a raw column following my ML-03 approach, I'll build a composite engagement opportunity score from a CTR-like piece and an engagement-like piece.

I deliberately exclude gsc_clicks and ga4_engaged_sessions from features, since these are the exact ingredients I'll use to build that label using them as features too would let the model see its own answer key, the same leakage risk flagged in ML-03 with ctr/engagement_rate, just under new column names in this table.

In [7]:
field_roles = {
    "report_date": "context", "client_hash_id": "context", "content_hash_id": "context", "month": "context",
    "client_has_gsc": "context", "client_has_ga4": "context",
    "gsc_data_available": "context", "ga4_data_available": "context",
    "gsc_impressions": "feature", "gsc_clicks": "excluded (label ingredient)",
    "gsc_sum_position": "feature", "gsc_avg_position": "feature",
    "ga4_pageviews": "feature", "ga4_sessions": "feature", "ga4_users": "feature",
    "ga4_engaged_sessions": "excluded (label ingredient)", "ga4_total_engagement_sec": "feature",
    "sessions_organic": "feature", "sessions_direct": "feature", "sessions_referral": "feature",
    "sessions_social": "feature", "sessions_paid": "feature", "sessions_ai": "feature",
    "ai_chatgpt": "feature", "ai_perplexity": "feature", "ai_gemini": "feature",
    "ai_copilot": "feature", "ai_claude": "feature", "ai_meta": "feature", "ai_other": "feature",
    "scroll_events": "feature"
}

roles_df = pd.DataFrame(list(field_roles.items()), columns=["column", "role"])
print(roles_df.sort_values("role").to_string(index=False))

Shape: (104, 9)
Columns: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']
Total clients: 104
Clients with GSC data available by March 2026: 52
Clients with GA4 data available by March 2026: 26

Missing gsc_data_start: 37
Missing ga4_data_start: 53

Earliest / latest gsc_data_start: 2025-01-27 / 2026-06-02
Earliest / latest ga4_data_start: 2025-10-29 / 2026-06-01
Clients with GA4 ready by March 2026: 26
Earliest GA4 start among THOSE clients: 2025-10-29


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [17]:
#PART A

# Missing values check: does ga4_total_engagement_sec go missing exactly where GA4 isn't available?
# Missing values check: are "not available" GA4 rows blank, or secretly zero-filled?
missing_engagement = df_march["ga4_total_engagement_sec"].isna().sum()
not_available = (df_march["ga4_data_available"] != True).sum()
zero_but_flagged_unavailable = df_march[
    (df_march["ga4_data_available"] != True) & (df_march["ga4_total_engagement_sec"] == 0)
].shape[0]

print("Rows with missing (NaN) ga4_total_engagement_sec:", missing_engagement)
print("Rows where ga4_data_available is NOT True:", not_available)
print("Rows that are NOT available but show a FAKE ZERO (not NaN):", zero_but_flagged_unavailable)



Rows with missing (NaN) ga4_total_engagement_sec: 3018741
Rows where ga4_data_available is NOT True: 9427412
Rows that are NOT available but show a FAKE ZERO (not NaN): 6408671


I checked whether missing values and the availability flag line up — they don't. Only 3,018,741 rows have a truly blank ga4_total_engagement_sec, but 9,427,412 rows are flagged as GA4-not-available, and 6,408,671 of those show a fake zero instead of a blank. This confirms the SKILL.md warning: unavailable GA4 rows are zero-filled, not left blank. Checking for NaN alone would badly undercount how much of my data is untrustworthy — the ga4_data_available flag is the only safe way to filter.

In [16]:

# PART B
# ============================================================
# BACKUP CODE FOR PART 4 LIMITATION: GA4/GSC Coverage BY MARCH 2026
# ============================================================

total_clients = df_clients.shape[0]
march_start = datetime.date(2026, 3, 1)

# Ready SPECIFICALLY by March 2026 (matches the written limitation)
ga4_ready_march = (df_clients["ga4_data_start"] <= march_start).sum()
ga4_not_ready_march = total_clients - ga4_ready_march

gsc_ready_march = (df_clients["gsc_data_start"] <= march_start).sum()

earliest_ga4_relevant = df_clients[df_clients["ga4_data_start"] <= march_start]["ga4_data_start"].min()

pct_ga4_available = round((ga4_ready_march / total_clients) * 100, 1)

print("Total clients:", total_clients)
print(f"Clients WITH GA4 ready by March 2026: {ga4_ready_march}")
print(f"Clients WITHOUT GA4 ready by March 2026: {ga4_not_ready_march}")
print(f"Clients WITH GSC ready by March 2026: {gsc_ready_march}")
print(f"Earliest GA4 start among March-ready clients: {earliest_ga4_relevant}")
print(f"Percent of clients with usable GA4 data by March: {pct_ga4_available}%")

Total clients: 104
Clients WITH GA4 ready by March 2026: 26
Clients WITHOUT GA4 ready by March 2026: 78
Clients WITH GSC ready by March 2026: 52
Earliest GA4 start among March-ready clients: 2025-10-29
Percent of clients with usable GA4 data by March: 25.0%


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Out of 104 total clients, only 26 clients have GA4 access at all , the rest, 53 clients, have no GA4 tracking connected. On top of that, even for the clients that do have GA4, tracking only starts as early as October 2025, so history is short.

This means my full engagement score (which needs both CTR and engagement rate) can only really be trusted for around a quarter of my clients. For the remaining clients, I only have GSC data, so I can only score them on CTR, not full engagement.

Because of this, I am scoping my engagement scoring to clients who have GA4 access. For the rest, only a CTR-based signal will be available, and I will be upfront about this gap instead of pretending the score means the same thing for everyone.

In [15]:
# Backup evidence for Section 4 limitation (same result as Section 3, Part B — shown here for convenience)
total_clients = df_clients.shape[0]
march_start = datetime.date(2026, 3, 1)

ga4_ready_march = (df_clients["ga4_data_start"] <= march_start).sum()
ga4_not_ready_march = total_clients - ga4_ready_march
gsc_ready_march = (df_clients["gsc_data_start"] <= march_start).sum()
earliest_ga4_relevant = df_clients[df_clients["ga4_data_start"] <= march_start]["ga4_data_start"].min()
pct_ga4_available = round((ga4_ready_march / total_clients) * 100, 1)

print("Total clients:", total_clients)
print(f"Clients WITH GA4 ready by March 2026: {ga4_ready_march}")
print(f"Clients WITHOUT GA4 ready by March 2026: {ga4_not_ready_march}")
print(f"Percent of clients with usable GA4 data by March: {pct_ga4_available}%")

Total clients: 104
Clients WITH GA4 ready by March 2026: 26
Clients WITHOUT GA4 ready by March 2026: 78
Percent of clients with usable GA4 data by March: 25.0%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.